Review 2026-09-08: illustrative policy only. Forecast error inputs depend on withdrawn legacy evaluation. Refit and validate SKU-store errors before interpreting stock levels. See ../DECISION_REVIEW.md.

# 03 · Inventory Optimization — From forecast to decision

This is where the forecast and its uncertainty become operational decisions:
safety stock, reorder point, ABC prioritization and the service-level vs cost
trade-off.

**The chain:** a better forecast → lower error (σ) → **less safety stock for the
same service level** → less capital frozen and less spoilage.

The formulas are deterministic given the forecast and its error, so they are
reused from `src/forecasting_pipeline.py`.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, os
sys.path.insert(0, "../src")   # to import forecasting_pipeline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from favorita_utils import q, apply_plot_style, COLORS
from forecasting_pipeline import (
    Config, safety_stock, reorder_point, build_inventory_policy,
    service_level_tradeoff, abc_classification,
)
apply_plot_style()
cfg = Config(lead_time=3, service_level=0.95)
print(f"Config: lead_time={cfg.lead_time}d, base service level={cfg.service_level}")

## 1. Inputs: forecast error + ABC classification

The per-item forecast error comes from notebook 02. The ABC classification comes
from the dbt mart (`mart_item_abc`), rather than recomputing it.

In [ ]:
# Per-item forecast error (generated in 02_forecasting.ipynb)
err = pd.read_csv("../outputs/forecast_error_by_item.csv", encoding="utf-8-sig")
print(f"Forecast error: {len(err)} items")

# ABC from the dbt mart
abc = q('''select item_nbr, abc_class as abc, total_sales
           from main_marts.mart_item_abc''')
print(f"ABC from dbt: {len(abc)} items")

# Join
base = err.merge(abc[["item_nbr", "abc"]], on="item_nbr", how="left")
base["abc"] = base["abc"].fillna("C")   # unclassified items -> conservative C
base.head().round(2)

## 2. Safety stock: the buffer against uncertainty

$$ \text{safety stock} = z \times \sigma_{\text{error}} \times \sqrt{\text{lead time}} $$

Where `z` is the service-level factor (95% → 1.65; 99% → 2.33) and `σ_error` is
the standard deviation of the forecast error (from backtesting). We show how the
same item needs a different buffer depending on the required service level.

In [ ]:
example = base.iloc[0]
print(f"Item {int(example.item_nbr)} — σ_error={example.sigma_error:.2f}, "
      f"mean demand={example.avg_demand:.1f}/day\n")
print("Service level |  z    | safety stock | reorder point")
print("-" * 55)
for sl in (0.90, 0.95, 0.98, 0.99):
    ss = safety_stock(example.sigma_error, cfg.lead_time, sl, cfg)
    rop = reorder_point(example.avg_demand, cfg.lead_time, ss)
    z = cfg.z_scores[round(sl, 2)]
    print(f"     {sl:.0%}      | {z:.3f} |    {ss:6.1f}    |     {rop:6.1f}")

**Reading:** going from 95% to 99% service level requires more buffer, and how
much more depends on σ. An item with a precise forecast (low σ) pays little for
the extra level; an unpredictable one pays a lot.

## 3. Full inventory policy

We build the policy for all items, with **service level differentiated by ABC
class** (A stricter) and a half-step extra for perishables (they can't be
over-ordered → less tolerance for stockout).

In [ ]:
# build_inventory_policy expects columns: item_nbr, sigma_error, avg_demand, perishable
policy = build_inventory_policy(
    forecast_error_by_item=base[["item_nbr", "sigma_error", "avg_demand", "perishable"]],
    abc=base[["item_nbr", "abc"]],
    cfg=cfg,
)
print(f"Policy built for {len(policy)} items")
policy.head(10)

In [ ]:
# Save the policy — a key deliverable of the project
os.makedirs("../outputs", exist_ok=True)
policy.to_csv("../outputs/inventory_policy.csv", index=False, encoding="utf-8-sig")
print("Saved outputs/inventory_policy.csv")

# Summary by class
class_summary = (policy.groupby("abc")
                 .agg(n_items=("item_nbr", "count"),
                      service_level=("service_level", "mean"),
                      mean_safety_stock=("safety_stock", "mean"),
                      mean_rop=("reorder_point", "mean"))
                 .reset_index())
class_summary.round(2)

**Reading:** class A gets the highest service level and, consequently, more
safety stock; class C is managed with less buffer. A uniform service level would
over-protect the tail and under-protect the head.

## 4. Safety stock distribution by class

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Boxplot of safety stock by class
data_by_class = [policy[policy.abc == c]["safety_stock"].values for c in ["A", "B", "C"]]
a1.boxplot(data_by_class, labels=["A", "B", "C"])
a1.set_title("Safety stock distribution by ABC class")
a1.set_ylabel("Safety stock units")

# Perishables vs non
per_data = [policy[policy.perishable == p]["service_level"].values for p in [0, 1]]
a2.boxplot(per_data, labels=["Non-perishable", "Perishable"])
a2.set_title("Service level: perishables vs non")
a2.set_ylabel("Service level")
plt.tight_layout(); plt.show()

## 5. The trade-off curve: service level vs cost

Higher service level means more safety stock and more holding cost, but less lost
sales. The curve shows the break-even point.

In [ ]:
# Representative item (class A, high demand) for the curve
item_a = base[base.abc == "A"].sort_values("avg_demand", ascending=False).iloc[0]
unit_holding_cost = 0.5   # cost of holding one unit in inventory ($/unit)

curve = service_level_tradeoff(
    sigma_error=item_a.sigma_error,
    avg_demand=item_a.avg_demand,
    holding_cost=unit_holding_cost,
    cfg=cfg,
)
# Approximate lost-sales cost: proportional to (1 - service level)
sale_price = 3.0
curve["expected_lost_sales"] = (1 - curve["service_level"]) * item_a.avg_demand * sale_price * cfg.lead_time
curve["total_cost"] = curve["holding_cost"] + curve["expected_lost_sales"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(curve["service_level"], curve["holding_cost"], "o-", color=COLORS["primary"], label="Inventory cost")
ax.plot(curve["service_level"], curve["expected_lost_sales"], "o-", color=COLORS["bad"], label="Expected lost sales")
ax.plot(curve["service_level"], curve["total_cost"], "o-", color="black", lw=2, label="Total cost")
opt = curve.loc[curve["total_cost"].idxmin()]
ax.axvline(opt["service_level"], color=COLORS["good"], ls="--", alpha=0.7,
           label=f"Optimum ≈ {opt['service_level']:.0%}")
ax.set_title(f"Service level vs cost trade-off — item {int(item_a.item_nbr)} (class A)")
ax.set_xlabel("Service level"); ax.set_ylabel("Cost ($)")
ax.legend()
plt.tight_layout(); plt.show()
curve.round(2)

**Reading:** total cost is a U-shape — there's a point where the savings from
avoiding stockouts stop compensating the cost of holding more inventory. That's the
economically optimal service level, and it's different for each item depending on
its σ, its demand and its costs. *Costs here are illustrative; with real margin and
spoilage data the curve calibrates to the specific business.*

## 6. Sensitivity: what a lower forecast error releases

If σ falls, how much safety stock — and capital — is released at an unchanged
service level?

In [ ]:
# Scenario: the best model reduces forecast error by 20% vs the baseline
sigma_reduction = 0.20
current_ss = policy["safety_stock"].sum()

improved_policy = build_inventory_policy(
    base.assign(sigma_error=base["sigma_error"] * (1 - sigma_reduction))
        [["item_nbr", "sigma_error", "avg_demand", "perishable"]],
    base[["item_nbr", "abc"]],
    cfg,
)
improved_ss = improved_policy["safety_stock"].sum()

print(f"Total safety stock (baseline model)   : {current_ss:,.0f} units")
print(f"Total safety stock (-20% error)       : {improved_ss:,.0f} units")
print(f"Safety-stock reduction                : {current_ss - improved_ss:,.0f} units "
      f"({1 - improved_ss/current_ss:.1%})")
print()
print("-> Capital freed and spoilage avoided, at an unchanged service level.")
print("   Safety stock is linear in sigma, so the buffer tracks the error 1:1.")

---

## Summary

1. **Business question:** how much to order to avoid both stockouts and spoilage.
2. **Data modeling** (star schema in dbt) → **EDA** (notebook 01).
3. **Forecasting with temporal validation** (notebook 02): baseline → Prophet →
   SARIMAX → LightGBM, all beating the baseline.
4. **Translation to decision** (this notebook): safety stock, reorder point, ABC
   prioritization, service-level vs cost trade-off.
5. **Sensitivity:** safety stock is linear in σ, so a lower forecast error
   releases buffer proportionally at an unchanged service level.

**Generated deliverables:**
- `outputs/inventory_policy.csv` — per-item replenishment policy.
- `outputs/forecast_error_by_item.csv` — uncertainty input (from notebook 02).
